In [1]:
from sentence_transformers import SentenceTransformer

In [3]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [8]:
#Initialize the model
model=SentenceTransformer('all-MiniLM-L6-v2')

#simple text
text="""
The old clock in the hallway ticked louder than usual that night.
A stray cat followed me all the way to the bookstore.
The rain painted silver streaks across the windowpane.
She found a crumpled letter hidden inside the drawer.
The mountain air carried a silence that felt endless.

"""

#Step1:Splitting into sentences
sentences=[s.strip() for s in text.split("\n") if s.strip()]

In [9]:
#step 2:Embedding of each senteces
embeddings=model.encode(sentences)

In [10]:
threshold=0.7
chunks=[]
current_chunk=[sentences[0]]

In [12]:
#Step-4Semantic grouping based on threshold
for i in range(1,len(sentences)):
    sim=cosine_similarity(
        [embeddings[i-1]],[embeddings[i]]
    )[0][0]
    if sim>=threshold:
        current_chunk.append(sentences[i])
    else:
        chunks.append("".join(current_chunk))
        current_chunk=[sentences[i]]

#Append the last chunk
chunks.append("".join(current_chunk))
 

In [13]:
#Output of the above chunks
print("Semantic chunks")
for idx,chunk in enumerate(chunks):
    print(f"\n chunk {idx+1}:\n{chunk}")

Semantic chunks

 chunk 1:
The old clock in the hallway ticked louder than usual that night.

 chunk 2:
A stray cat followed me all the way to the bookstore.

 chunk 3:
The rain painted silver streaks across the windowpane.

 chunk 4:
She found a crumpled letter hidden inside the drawer.

 chunk 5:
The mountain air carried a silence that felt endless.


In [14]:
#Rag pipeline Modular Coding
import os

In [15]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [9]:
#Custom semantic chunker with threshold
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

class ThresholdSemanticChunker:
    def __init__(self,model_name="all-MiniLM-L6-v2",threshold=0.7):
        self.model=SentenceTransformer(
            model_name
        )
        self.threshold=threshold

    def split(self,text:str):
        sentences=[s.strip() for s in text.split("\n") if s.strip()]
        embeddings=self.model.encode(sentences)
        chunks=[]
        current_chunk=[sentences[0]]

        for i in range(1,len(sentences)):
            sim=cosine_similarity([embeddings[i-1]],[embeddings[i]])[0][0]
            if sim>=self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(".".join(current_chunk)+".")
                current_chunk=[sentences[i]]
        chunks.append(".".join(current_chunk)+".")
        return chunks

    def split_documents(self, docs):
        result = []
        for doc in docs:
            chunks = self.split(doc.page_content)
            for chunk in chunks:
                result.append(Document(page_content=chunk, metadata=doc.metadata))
        return result

In [10]:
#Sample text
sample_text="""
Langchain is a framework for building applciation with LLM'set
Langchain provides modular abstractions to combine LLMS with tools like OPENAI and pinecone
You can create chains,agents,memory,and retriever
The Eiffel tower si located in Paris.
France is a popular tourist destination
"""
doc=Document(page_content=sample_text)
doc

Document(metadata={}, page_content="\nLangchain is a framework for building applciation with LLM'set\nLangchain provides modular abstractions to combine LLMS with tools like OPENAI and pinecone\nYou can create chains,agents,memory,and retriever\nThe Eiffel tower si located in Paris.\nFrance is a popular tourist destination\n")

In [32]:
#CHunking

chunker=ThresholdSemanticChunker(threshold=0.7)
chunks=chunker.split_documents([doc])
chunks

[Document(metadata={}, page_content="Langchain is a framework for building applciation with LLM'set.Langchain provides modular abstractions to combine LLMS with tools like OPENAI and pinecone."),
 Document(metadata={}, page_content='You can create chains,agents,memory,and retriever.'),
 Document(metadata={}, page_content='The Eiffel tower si located in Paris..'),
 Document(metadata={}, page_content='France is a popular tourist destination.')]

In [36]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(
    model='sentence-transformers/all-MiniLM-L6-v2'
)
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [45]:
#Vector store
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)
retrieve=vectorstore.as_retriever()

In [46]:
print(type(embeddings))
print(embeddings)

<class 'langchain_huggingface.embeddings.huggingface.HuggingFaceEmbeddings'>
model_name='sentence-transformers/all-MiniLM-L6-v2' cache_folder=None model_kwargs={} encode_kwargs={} query_encode_kwargs={} multi_process=False show_progress=False


In [47]:
#Prompt Template
#5-The prompt template over here is
from langchain_core.prompts import PromptTemplate
template="""Answer the question based on the follwing context:
{context}
Question:{question}
"""
prompt=PromptTemplate.from_template(template)

In [ ]:
#LLM here is
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama3-8b-8192", temperature=0.4)

ValueError: Unable to infer model provider for model='openai/gpt-oss-120b', please specify model_provider directly.

In [55]:
#LCEL Chain with RETRIEVAL
from langchain_core.runnables import RunnableMap
from langchain_core.output_parsers import StrOutputParser
rag_chain=(
    RunnableMap(
        {
            "context":lambda x: retrieve.invoke(x['question']),
            "question":lambda x:x['question'],

        }
    )
    | prompt
    | llm
    | StrOutputParser()
)
query={"question":"What is LangChain used for?"}
result=rag_chain.invoke(query)
print(result)

BadRequestError: Error code: 400 - {'error': {'message': 'The model `gemma2-9b-it` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}